# Augmented classifier: training, testing, running

Same pipeline as `baseline.ipynb` — fine-tunes Ultralytics YOLO
classification (`yolo26n-cls.pt`) on SID-Set for real vs
AI-generated/tampered — but trains on **realistically degraded** images
instead of clean ones.

Everything runs through the shared classes in
`packages/models/normal_classifier` (`NormalClassifierTrainer`,
`NormalClassifierDetector`).

The only real difference from the baseline is the data source:
`data.dataset_builder.augmented_sid_dataset()`, a re-iterable stream over
SID-Set that

- pulls a balanced subset from Hugging Face **without downloading the whole
  dataset**;
- decodes each image **off the streaming thread** and runs it through the
  data package's `ImageAugmenter` (JPEG compression, blur, resize, noise,
  colour jitter, centre crop), `num_augmentations=(2, 5)` per image, on a
  worker pool;
- **caches** each image's raw bytes to `sid_cache/` on the first pass, so
  `evaluate()`, re-runs and `baseline.ipynb` read from local disk instead
  of the network;
- re-streams on each iteration (seed-deterministic), so one object backs
  `train()`'s validation pass, `evaluate()` and the inference demo.

The held-out set is streamed **clean** (`augment=False`) so metrics measure
generalisation rather than robustness to a fixed corruption.


## 1. Setup — clone the repo, install deps, wire up imports

In [ ]:
%cd /content/
!git clone https://github.com/Zhongbob/TikTokTechJam2026.git

In [ ]:
%cd /content/TikTokTechJam2026
!git switch "setup"
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os,sys
os.environ["PATH"] += f":{os.path.expanduser('~/.local/bin')}"
os.environ["UV_PROJECT_ENVIRONMENT"] = sys.prefix
%cd /content/TikTokTechJam2026/packages/models/normal_classifier
!uv sync


In [ ]:
!uv pip install --system ../../data
# Might need to install local packages manually if the above doesnt work

# RESTART SESSION UNDER RUNTIME AFTER COMPLETING THE ABOVE

In [ ]:
# Run this AFTER restarting the session (cell above). Locates the .venv
# `uv sync` created and bridges its site-packages into this fresh kernel
# process via site.addsitedir() -- the API that actually processes uv's
# editable-install ".pth" redirects; a plain sys.path.insert() does not.
import glob
import importlib
import site
from pathlib import Path

candidates = [
    Path("/content/TikTokTechJam2026/.venv"),
    Path("/content/TikTokTechJam2026/packages/models/normal_classifier/.venv"),
]
venv_dir = next((c for c in candidates if c.is_dir()), None)
if venv_dir is None:
    found = glob.glob("/content/TikTokTechJam2026/**/.venv", recursive=True)
    assert found, "no .venv found under the repo -- did `uv sync` in the cell above actually succeed?"
    venv_dir = Path(found[0])

site_packages = next(venv_dir.glob("lib/python*/site-packages"))
print(f"bridging {site_packages}")
site.addsitedir(str(site_packages))

importlib.invalidate_caches()
importlib.import_module("normal_classifier")
print("'normal_classifier' imported OK")


## 2. Stream augmented training + clean validation data from SID-Set

In [ ]:
import os

from data.dataset_builder import augmented_sid_dataset

# Re-iterable, memory-bounded streams. Speed-ups over a naive stream:
#   * decode off the streaming thread -- the source is raw image bytes, so
#     PNG/JPEG decoding + resize + the transform chain all run on the worker
#     pool (this is the main win; a plain stream decodes serially).
#   * backend="thread" (default) parallelises that pool -- Pillow/NumPy
#     release the GIL. num_workers defaults to CPU count.
#   * cache_dir -> the first pass writes each image's *raw bytes* to local
#     disk; later passes (train's val pass, evaluate(), re-runs, and
#     baseline.ipynb) read bytes from disk instead of Hugging Face. Point it
#     at /content/drive/MyDrive/sid_cache to survive runtime recycles.
WORKERS = os.cpu_count()

# Training stream: each image gets a random 2-5 of the data package's six
# corruptions (JPEG, blur, resize, noise, colour jitter, crop).
train_samples = augmented_sid_dataset(
    images_per_label=1000, split="train",
    num_augmentations=(2, 5), output_size=(224, 224),
    num_workers=WORKERS, cache_dir="sid_cache",
)

# Validation stream: same source, left clean (augment=False). Shares the
# sid_cache/ entry with baseline.ipynb.
val_samples = augmented_sid_dataset(
    images_per_label=200, split="validation",
    augment=False, output_size=(224, 224),
    num_workers=WORKERS, cache_dir="sid_cache",
)

print(f"train: {train_samples}  (~{len(train_samples)} samples)")
print(f"val:   {val_samples}  (~{len(val_samples)} samples)")


In [ ]:
# Warm the disk cache up front: one pass that just downloads each image's raw
# bytes (no decoding) with a progress bar. This isolates the network cost --
# after it, train() and evaluate() read bytes from sid_cache/ and the per-image
# decode + augmentation happens in parallel on the worker pool.
train_samples.warm_cache()
val_samples.warm_cache()


## 3. Train

`NormalClassifierTrainer.train()` exports `train_samples`/`val_samples` into
the `real/`, `ai_generated/` class-folder layout Ultralytics' classification
trainer expects, then fine-tunes `yolo26n-cls.pt` on them. Each source image
is augmented as it is streamed, then written once and reused across epochs.

In [ ]:
from normal_classifier import NormalClassifierTrainer

trainer = NormalClassifierTrainer(base_weights="yolo26n-cls.pt", image_size=224)
result = trainer.train(
    train_samples,
    val_samples=val_samples,
    output_dir="SID_YOLO_AUG",
    epochs=100,
    batch=32,
    patience=10,
    device="cpu",
    plots=True,
)
result


## 4. Test

The "testing" stage — score the trained model against a held-out set via
`.evaluate()`. SID-Set only exposes train/validation splits, so this
re-iterates `val_samples` (served from the `sid_cache/` byte cache after
step 3, still clean); swap in a separate held-out set here if you have one.


In [ ]:
metrics = trainer.evaluate(val_samples, output_dir="SID_YOLO_AUG_eval")
print("Held-out evaluation metrics:", metrics)


In [ ]:
trainer.save("normal_classifier_augmented.pt")
print("Saved checkpoint to normal_classifier_augmented.pt")


## 5. Run (inference)

The "running" stage — `NormalClassifierDetector` wraps the saved checkpoint
and implements the same `EnsembleDetector` contract `apps/web` consumes, so
this class can be dropped straight into
`apps/web/src/web/services/factory.py`'s `get_detector()` once ready.

In [ ]:
import itertools

from normal_classifier import NormalClassifierDetector

detector = NormalClassifierDetector.from_checkpoint("normal_classifier_augmented.pt")

# val_samples re-streams on iteration; islice pulls just the first 5, so only
# those 5 images are ever decoded here.
for sample in itertools.islice(val_samples, 5):
    detection = detector.predict(sample.image)
    print(
        f"true={sample.metadata['label_name']:<10} "
        f"predicted={detection.verdict:<12} "
        f"p(ai_generated)={detection.ai_generated_probability:.2f}"
    )


In [ ]:
# Score the deployment-ready detector on the held-out set and write a
# confusion matrix (detector_eval/confusion_matrix.png).
metrics = detector.evaluate(val_samples, generate_confusion_matrix=True, output_dir="detector_eval")
print(metrics)

from PIL import Image as _Image
_Image.open("detector_eval/confusion_matrix.png")
